# Evaluate ai.classify(...) Quality with PySpark

This notebook evaluates custom text classification with judge-derived reference labels. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Classify customer messages into a fixed category set.
2. Use a larger judge model to identify the expected category.
3. Calculate accuracy, macro precision, recall, F1, and per-class results.
4. Compare the baseline with a custom structured classifier.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Accuracy** | Overall category correctness |
| **Precision** | Correct predictions within each predicted category |
| **Recall** | Coverage of each expected category |
| **F1 score** | Balance of precision and recall |

[ai.classify PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/classify)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The executor uses `gpt-5-mini` with low reasoning effort. The judge uses
`gpt-5.1` with medium reasoning effort and remains fixed across comparisons.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",
    "reasoningEffort": "medium",
}

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")


## 2. Load Sample Data


In [ ]:
CATEGORY_GUIDANCE = {'technical_support': 'Product behavior, bugs, integration failures, or how-to issues', 'billing': 'Charges, refunds, invoices, subscriptions, or pricing', 'feedback': 'Compliments, complaints, or feature suggestions', 'general_inquiry': 'Product, service, or company information'}
CATEGORIES = list(CATEGORY_GUIDANCE)
rows = [(1,
  'I reset my password and entered the correct 2FA code, but the login page keeps reloading instead of '
  'signing me in. I tested in Chrome and Safari and got the same result.'),
 (2,
  'I was charged twice this week for the same monthly plan. Please refund the duplicate $49.99 charge and '
  'send me an itemized invoice for the last three months.'),
 (3,
  'Feature request: please add a bulk export option for reports. Downloading 30 client reports one by one at '
  'quarter end takes too much time.'),
 (4,
  'We are considering your platform for about 500 employees. Can you share enterprise feature differences, '
  'SSO support, implementation timeline, and whether a dedicated account manager is included?'),
 (5,
  "Our integration started returning HTTP 504 errors on /v2/batch-process after last week's update. Payloads "
  'above 5 MB time out around 30 seconds in about 40% of requests.'),
 (6,
  'Kudos to the onboarding team - they helped us migrate data and stayed late before go-live. The support '
  'was clear and proactive throughout the rollout.'),
 (7,
  'Ticket #TKT-8842 is still unresolved after 72 hours while our production dashboard is down. Premium SLA '
  'says Sev-1 responses should be within 4 hours - please escalate immediately.')]
df = spark.createDataFrame(rows, ["sample_id", "text"])
df = df.withColumn("_category_guidance", F.lit("\n".join(f"- {name}: {description}" for name, description in CATEGORY_GUIDANCE.items())))
display(df.select("sample_id", "text"))


## 3. Run `ai.classify`


In [ ]:
classified_df = materialize(
    df.ai.classify(
        labels=CATEGORIES,
        input_col="text",
        output_col="category",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(classified_df.select("text", "category"))
display(classified_df.ai.stats)


## 4. Evaluate with an LLM Judge


In [ ]:
from typing import Literal

Category = Literal[tuple(CATEGORIES)]

class ClassifyEval(BaseModel):
    reason: str = Field(description="Brief rationale for the expected category")
    expected_category: Category = Field(description="Best category from the provided list")

EVAL_PROMPT = """Evaluate whether the predicted category is the best choice.

<category_definitions>
{_category_guidance}
</category_definitions>
<text>
{text}
</text>
<predicted_category>
{category}
</predicted_category>

Focus on the primary intent. Return a brief rationale and the best expected category."""


In [ ]:
evaluated_df = fresh_ai_view(classified_df).ai.generate_response(
    prompt=EVAL_PROMPT,
    is_prompt_template=True,
    output_col="_eval_response",
    error_col="_eval_error",
    response_format=ClassifyEval,
    **JUDGE_OPTIONS,
)
evaluated_df = (
    evaluated_df
    .withColumn(
        "expected_category",
        F.get_json_object(F.col("_eval_response"), "$.expected_category"),
    )
    .withColumn(
        "correct",
        F.col("category") == F.col("expected_category"),
    )
    .withColumn(
        "eval_reason",
        F.get_json_object(F.col("_eval_response"), "$.reason"),
    )
)
evaluated_df = materialize(evaluated_df)
display(
    evaluated_df.select(
        "text", "category", "expected_category", "correct", "eval_reason"
    )
)


## 5. Results


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

results_pd = evaluated_df.select(
    "sample_id", "text", "category", "expected_category", "correct", "eval_reason"
).toPandas()
metric_input = results_pd.dropna(subset=["expected_category", "category"]).copy()
excluded_rows = len(results_pd) - len(metric_input)
if excluded_rows:
    print(f"Excluded {excluded_rows} row(s) without both expected and predicted categories.")
if metric_input.empty:
    raise ValueError("No valid classification rows are available for metric calculation.")
y_true = metric_input["expected_category"]
y_pred = metric_input["category"]

metric_names = ["Accuracy", "Precision", "Recall", "F1 score"]
metric_values = [
    accuracy_score(y_true, y_pred),
    precision_score(y_true, y_pred, average="macro", zero_division=0),
    recall_score(y_true, y_pred, average="macro", zero_division=0),
    f1_score(y_true, y_pred, average="macro", zero_division=0),
]
metrics_pd = pd.DataFrame({"Metric": metric_names, "Score": metric_values})
display(metrics_pd.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(
    metric_names,
    metric_values,
    color=["#0077aa", "#22cc77", "#9955bb", "#ee7722"],
)
axes[0].set_ylim(0, 1)
axes[0].set_title("Classification Metrics")
axes[0].axhline(y=0.8, color="#999999", linestyle="--", alpha=0.6)
axes[0].bar_label(bars, fmt="%.2f", padding=2)

correct_count = int((y_true == y_pred).sum())
axes[1].pie(
    [correct_count, len(metric_input) - correct_count],
    labels=["Correct", "Incorrect"],
    autopct="%1.0f%%",
    colors=["#22cc77", "#ee4433"],
)
axes[1].set_title("Judge-Derived Accuracy")
plt.tight_layout()
plt.show()

report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
labels = sorted(set(y_true) | set(y_pred))
per_class_pd = pd.DataFrame([
    {
        "category": label,
        "precision": report[label]["precision"],
        "recall": report[label]["recall"],
        "f1_score": report[label]["f1-score"],
        "support": int(report[label]["support"]),
    }
    for label in labels
    if label in report
])
display(per_class_pd.round(3))


In [ ]:
incorrect_pd = results_pd.loc[
    results_pd["correct"].ne(True),
    ["text", "category", "expected_category", "eval_reason"],
].copy()
if incorrect_pd.empty:
    incorrect_pd = pd.DataFrame([{
        "text": "All predictions matched the judge-derived labels.",
        "category": "",
        "expected_category": "",
        "eval_reason": "",
    }])
else:
    incorrect_pd["text"] = incorrect_pd["text"].str[:120] + "..."
display(incorrect_pd)


## 6. Optional Refinement: Custom Structured Classifier

Compare the tuned `ai.classify` baseline with a domain-specific structured
prompt while reusing the same judge-derived expected categories.


In [ ]:
class CustomClassifyResult(BaseModel):
    reason: str = Field(description="Brief rationale for the category")
    category: Category = Field(description="Best category from the provided list")

CUSTOM_PROMPT = """Classify the customer message into exactly one category.

<category_definitions>
{_category_guidance}
</category_definitions>

<text>
{text}
</text>"""

custom_df = fresh_ai_view(evaluated_df).ai.generate_response(
    prompt=CUSTOM_PROMPT,
    is_prompt_template=True,
    output_col="_custom_response",
    error_col="_custom_error",
    response_format=CustomClassifyResult,
    **EXECUTOR_OPTIONS,
)
custom_df = (
    custom_df
    .withColumn(
        "custom_category",
        F.trim(F.get_json_object(F.col("_custom_response"), "$.category")),
    )
    .withColumn(
        "custom_reason",
        F.get_json_object(F.col("_custom_response"), "$.reason"),
    )
)
custom_df = materialize(custom_df)
display(
    custom_df.select(
        "text", "category", "expected_category", "custom_category", "custom_reason"
    )
)


In [ ]:
custom_pd = custom_df.select(
    "sample_id", "expected_category", "category", "custom_category"
).toPandas()
required_columns = ["expected_category", "category", "custom_category"]
paired_mask = custom_pd[required_columns].notna().all(axis=1)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(custom_pd.loc[~paired_mask, ["sample_id", *required_columns]])
if not paired_count:
    raise ValueError("No rows have both baseline and custom classification results.")

paired_pd = custom_pd.loc[paired_mask]
expected = paired_pd["expected_category"]

def classification_metrics(column_name):
    predictions = paired_pd[column_name]
    return [
        accuracy_score(expected, predictions),
        precision_score(expected, predictions, average="macro", zero_division=0),
        recall_score(expected, predictions, average="macro", zero_division=0),
        f1_score(expected, predictions, average="macro", zero_division=0),
    ]

comparison_pd = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 score"],
    "Baseline": classification_metrics("category"),
    "Custom": classification_metrics("custom_category"),
})
comparison_pd["Delta"] = comparison_pd["Custom"] - comparison_pd["Baseline"]
display(comparison_pd.round(3))

ax = comparison_pd.set_index("Metric")[["Baseline", "Custom"]].plot.bar(
    figsize=(8, 4),
    color=["#0077aa", "#22cc77"],
    rot=0,
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Baseline vs Custom Classifier")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2)
plt.tight_layout()
plt.show()


## Interpreting Results

| Metric score | Suggested action |
|--------------|------------------|
| **0.80-1.00** | Strong result; review per-class metrics and flagged samples |
| **0.70-0.79** | Good starting point; investigate weaker classes |
| **Below 0.70** | Refine labels, data coverage, or the executor approach |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Low precision | Category boundaries overlap | Clarify or merge similar categories |
| Low recall | The category is underrepresented | Add representative examples |
| Low overall accuracy | Primary intent is ambiguous | Refine category definitions and routing rules |

### Improving Quality

- Make category definitions mutually exclusive and include representative edge cases.
- Merge categories that humans cannot label consistently.
- For harder classification, test `gpt-5.1` with medium reasoning effort as the
  executor while keeping the judge unchanged.

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.classify PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/classify)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
